# Close singular values: rank-three kernel recovery

Representative fit and Figures 1--2 of the synthetic recovery block. Execute a partir da raiz `code/fsnm`. O notebook grava figuras apenas em `code/fsnm/figures`; a cópia para o diretório TeX é deliberadamente manual.

## Data-generating process and evaluation helpers

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from fsnm import empirical_loss, fit_fsnm


def basis_matrix(values, rank=3):
    values = np.asarray(values)
    basis = np.column_stack(
        [
            np.sqrt(2) * np.sin(np.pi * values),
            np.sqrt(2) * np.cos(np.pi * values),
            np.sqrt(2) * np.sin(2 * np.pi * values),
        ]
    )
    return basis[:, :rank]


def kappa_exact(x_values, y_values):
    return 1 + (basis_matrix(x_values) * SIGMAS) @ basis_matrix(y_values).T


def sample_joint(size, seed):
    rng = np.random.default_rng(seed)
    upper_bound = 1 + 2 * SIGMAS.sum()
    x_parts = []
    y_parts = []
    n_accepted = 0

    while n_accepted < size:
        x = rng.uniform(-1, 1, size)
        y = rng.uniform(-1, 1, size)
        density_ratio = 1 + np.sum(
            basis_matrix(x) * SIGMAS * basis_matrix(y), axis=1
        )
        accepted = rng.uniform(size=size) < density_ratio / upper_bound
        x_parts.append(x[accepted])
        y_parts.append(y[accepted])
        n_accepted += accepted.sum()

    return np.concatenate(x_parts)[:size], np.concatenate(y_parts)[:size]


def scaled_factors(phi_model, psi_model, singular_values, x, y):
    scale = np.sqrt(np.maximum(singular_values, 0))
    return phi_model.predict(x[:, None]) * scale, psi_model.predict(y[:, None]) * scale


def subspace_error(estimated, exact):
    estimated = estimated - estimated.mean(axis=0)
    exact = exact - exact.mean(axis=0)
    estimated_basis = np.linalg.qr(estimated)[0][:, : exact.shape[1]]
    exact_basis = np.linalg.qr(exact)[0][:, : exact.shape[1]]
    difference = estimated_basis @ estimated_basis.T - exact_basis @ exact_basis.T
    return np.linalg.norm(difference, ord="fro") / np.sqrt(2 * exact.shape[1])


def orthogonality_error(values):
    centered = values - values.mean(axis=0)
    gram = centered.T @ centered / len(centered)
    return np.linalg.norm(gram - np.eye(gram.shape[0]), ord="fro") / np.sqrt(gram.shape[0])


SIGMAS = np.array([0.18, 0.16, 0.12])
RANK = len(SIGMAS)
N_TRAIN = 10_000
N_VALIDATION = 4_000
LOSS_CURVE_ITERATIONS = 40
SEED = 0


## Fit the selected model and compute the reported metrics

In [2]:

x_train, y_train = sample_joint(N_TRAIN, seed=12)
x_validation, y_validation = sample_joint(N_VALIDATION, seed=99)

selected = dict(
    rank=RANK, n_iterations=11, step_size=0.1,
    max_depth=3, min_samples_leaf=300, seed=SEED,
)
phi_fsnm, psi_fsnm, values_fsnm, _ = fit_fsnm(
    x_train, y_train, **selected,
)
_, _, _, history_fsnm = fit_fsnm(
    x_train, y_train, validation_data=(x_validation, y_validation),
    **{**selected, "n_iterations": 40},
)

phi_validation, psi_validation = scaled_factors(
    phi_fsnm, psi_fsnm, values_fsnm, x_validation, y_validation,
)
validation_loss = float(empirical_loss(phi_validation, psi_validation))
grid = np.linspace(-1, 1, 160)
exact_basis = basis_matrix(grid)
kappa_true = kappa_exact(grid, grid)
phi_grid = phi_fsnm.predict(grid[:, None])
psi_grid = psi_fsnm.predict(grid[:, None])
kappa_estimated = 1 + (phi_grid * values_fsnm) @ psi_grid.T

metrics = {
    "kernel RMSE": np.sqrt(np.mean((kappa_estimated - kappa_true) ** 2)),
    "spectrum error": np.linalg.norm(values_fsnm - SIGMAS),
    "subspace error": 0.5 * (
        subspace_error(phi_grid, exact_basis)
        + subspace_error(psi_grid, exact_basis)
    ),
    "orthogonality error": 0.5 * (
        orthogonality_error(phi_grid) + orthogonality_error(psi_grid)
    ),
}
print("singular values:", np.round(values_fsnm, 4))
print("validation loss:", round(validation_loss, 4))
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")


singular values: [0.169  0.1386 0.112 ]
validation loss: -0.0585
kernel RMSE: 0.1204
spectrum error: 0.0254
subspace error: 0.3340
orthogonality error: 0.0209


## Kernel and optimization figures

In [3]:
value_limits = (
    min(kappa_true.min(), kappa_estimated.min()),
    max(kappa_true.max(), kappa_estimated.max()),
)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
for axis, (values, title) in zip(
    axes[:2],
    [
        (kappa_true, r"True $\kappa$"),
        (kappa_estimated, r"FSNM $\widehat\kappa$"),
    ],
):
    image = axis.imshow(
        values.T,
        origin="lower",
        extent=[-1, 1, -1, 1],
        cmap="coolwarm",
        vmin=value_limits[0],
        vmax=value_limits[1],
    )
    axis.set(title=title, xlabel="$x$", ylabel="$y$")
    fig.colorbar(image, ax=axis, shrink=0.82)

error = kappa_estimated - kappa_true
error_limit = np.max(np.abs(error))
image = axes[2].imshow(
    error.T,
    origin="lower",
    extent=[-1, 1, -1, 1],
    cmap="coolwarm",
    vmin=-error_limit,
    vmax=error_limit,
)
axes[2].set(title=r"$\widehat\kappa-\kappa$", xlabel="$x$", ylabel="$y$")
fig.colorbar(image, ax=axes[2], shrink=0.82)

project_directory = Path.cwd()
figure_directory = project_directory / "figures"
figure_directory.mkdir(exist_ok=True)
figure_path = figure_directory / "00_rank3_close_spectrum.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {figure_path}")

iterations = np.arange(1, len(history_fsnm["training_loss"]) + 1)
loss_figure, loss_axis = plt.subplots(figsize=(5.5, 3.6), constrained_layout=True)
loss_axis.plot(
    iterations, history_fsnm["training_loss"],
    color="tab:blue", linewidth=2, label="Training",
)
loss_axis.plot(
    iterations, history_fsnm["validation_loss"],
    color="tab:orange", linewidth=2, label="Validation",
)
loss_axis.axvline(
    selected["n_iterations"], color="black",
    linestyle=":", linewidth=1.5, label="Selected iteration",
)
loss_axis.set(xlabel="Iteration", ylabel="Empirical loss")
loss_axis.grid(alpha=0.25)
loss_axis.margins(y=0.12)
loss_axis.legend(frameon=False)
loss_path = figure_directory / "00_rank3_training_loss.png"
loss_figure.savefig(loss_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {loss_path}")


Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_rank3_close_spectrum.png
Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_rank3_training_loss.png
